# Generate PDBs & Compute TM-scores

Run the OpenFold structure module on both **original** and **reconstructed** test `.npy` files,
then compute TM-scores to compare them.

**Everything runs in-process** (no subprocesses) to avoid Python path / environment issues.

## 0. Install dependencies & patch OpenFold

In [ ]:
!pip install ml_collections dm-tree biopython modelcif --quiet
# Uninstall broken deepspeed/triton if present
!pip uninstall deepspeed triton -y 2>/dev/null; echo "deps done"

In [ ]:
# Patch OpenFold primitives.py to make flash_attn optional
import os
from pathlib import Path

# Auto-detect OPENFOLD_REPO: directory containing openfold/model/primitives.py
OPENFOLD_REPO = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / "openfold" / "model" / "primitives.py").exists():
        OPENFOLD_REPO = p
        break
if OPENFOLD_REPO is None:
    OPENFOLD_REPO = Path("/storage/ice1/1/0/varisa3/AlphaFold_Autoencoder/openfold")  # fallback

prim_path = OPENFOLD_REPO / "openfold" / "model" / "primitives.py"

with open(prim_path) as f:
    lines = f.readlines()

new_lines = []
i = 0
while i < len(lines):
    line = lines[i]
    stripped = line.strip()
    # Skip broken try/except from any previous patch attempt
    if stripped == 'try:' and i + 1 < len(lines) and 'flash_attn' in lines[i + 1]:
        i += 1
        continue
    if stripped.startswith('except') and i + 1 < len(lines) and '= None' in lines[i + 1] and 'flash' not in stripped:
        if any('flash_attn' in lines[max(0,i-3)+j] for j in range(4)):
            i += 2
            continue
    # Wrap any bare "from flash_attn..." import
    if stripped.startswith('from flash_attn'):
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(f'{indent}try:\n')
        new_lines.append(f'{indent}    {stripped}\n')
        new_lines.append(f'{indent}except (ImportError, OSError):\n')
        if 'unpad_input' in stripped:
            new_lines.append(f'{indent}    unpad_input = None\n')
        elif 'flash_attn_unpadded_qkvpacked_func' in stripped:
            new_lines.append(f'{indent}    flash_attn_unpadded_qkvpacked_func = None\n')
        else:
            new_lines.append(f'{indent}    pass\n')
        i += 1
        continue
    new_lines.append(line)
    i += 1

with open(prim_path, 'w') as f:
    f.writelines(new_lines)
print(f'Patched {prim_path}')

## 1. Setup paths & imports

In [ ]:
import sys
import os
import re
import json
import urllib.request
from pathlib import Path

import numpy as np
import torch

# --- CONFIGURE PATHS (auto-detect openfold repo root) ---
BASE = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / "openfold" / "model").exists() and (p / "VizFoldAutoencoder").exists():
        BASE = p
        break
BASE = BASE or Path.cwd()
VIZFOLD = BASE / "VizFoldAutoencoder"
OPENFOLD_REPO = BASE

# Add openfold to Python path so we can import it directly
if str(OPENFOLD_REPO) not in sys.path:
    sys.path.insert(0, str(OPENFOLD_REPO))

# Find checkpoint
CHECKPOINT = None
for candidate in [
    OPENFOLD_REPO / "resources" / "openfold_params" / "finetuning_ptm_1.pt",
    OPENFOLD_REPO / "openfold" / "resources" / "openfold_params" / "finetuning_ptm_1.pt",
]:
    if candidate.exists():
        CHECKPOINT = str(candidate)
        break

ORIGINAL_NPY_DIR = VIZFOLD / "original_proteins_layer47"
RECON_NPY_DIR = VIZFOLD / "reconstructed_proteins_layer47"
ORIGINAL_PDB_DIR = VIZFOLD / "original_pdbs"
RECON_PDB_DIR = VIZFOLD / "reconstructed_pdbs"

DEVICE = "cpu"  # CPU avoids flash_attn/CUDA issues; structure module is lightweight

os.chdir(VIZFOLD)

print(f"Python:        {sys.executable}")
print(f"PyTorch:       {torch.__version__}")
print(f"CUDA:          {torch.cuda.is_available()}")
print(f"OpenFold repo: {OPENFOLD_REPO}  (exists: {OPENFOLD_REPO.exists()})")
print(f"Checkpoint:    {CHECKPOINT}")
print(f"Original .npy: {len(list(ORIGINAL_NPY_DIR.glob('*.npy')))} files")
print(f"Recon .npy:    {len(list(RECON_NPY_DIR.glob('*.npy')))} files")
print(f"Device:        {DEVICE}")

In [ ]:
# Import OpenFold (now that sys.path is set and primitives.py is patched)
from openfold.config import model_config
from openfold.model.model import AlphaFold
from openfold.np import protein, residue_constants
from openfold.utils.import_weights import import_openfold_weights_
from openfold.utils.feats import atom14_to_atom37

print("OpenFold imported successfully!")

## 2. Load model once (reuse for all proteins)

In [ ]:
config = model_config("model_1_ptm")
model = AlphaFold(config)

# Load checkpoint
d = torch.load(CHECKPOINT, map_location="cpu")
if "ema" in d:
    d = d["ema"]["params"]
elif "state_dict" in d:
    d = d["state_dict"]
import_openfold_weights_(model, d)
del d  # free memory

model = model.to(DEVICE)
model.eval()
print(f"Model loaded on {DEVICE}")

## 3. Helper functions

In [ ]:
def fetch_sequence(pdb_id: str, chain_id: str) -> str:
    """Fetch sequence from RCSB PDB."""
    pdb_lower = pdb_id.lower()
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_lower}"
    with urllib.request.urlopen(url, timeout=30) as r:
        entry = json.loads(r.read().decode())
    polymer_ids = entry.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
    for eid in polymer_ids:
        url2 = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_lower}/{eid}"
        with urllib.request.urlopen(url2, timeout=30) as r:
            entity = json.loads(r.read().decode())
        asym_ids = entity.get("rcsb_polymer_entity_container_identifiers", {}).get("asym_ids", [])
        if chain_id in asym_ids:
            seq = entity.get("entity_poly", {}).get("pdbx_seq_one_letter_code", "")
            if not seq:
                seq = entity.get("entity_poly", {}).get("pdbx_seq_one_letter_code_can", "")
            if seq:
                return re.sub(r'[^ACDEFGHIKLMNPQRSTVWY]', 'X', seq)
    raise ValueError(f"Chain {chain_id} not found in {pdb_id}")


def sequence_to_aatype(seq: str) -> np.ndarray:
    return np.array(
        [residue_constants.restype_order.get(c, residue_constants.restype_num) for c in seq.upper()],
        dtype=np.int64,
    )


def npy_to_pdb(npy_path: str, output_dir: str, pdb_id: str, chain_id: str):
    """Convert a single .npy pair rep to a PDB file using the loaded model."""
    pair = np.load(npy_path, allow_pickle=True)
    if pair.ndim == 4:
        pair = pair[0]
    n_res = pair.shape[0]
    c_z = pair.shape[2]
    if c_z < 128:
        pair = np.concatenate([pair, np.zeros((n_res, n_res, 128 - c_z), dtype=pair.dtype)], axis=2)
    elif c_z > 128:
        pair = pair[:, :, :128].copy()

    # Fetch sequence
    seq = fetch_sequence(pdb_id, chain_id)
    if len(seq) != n_res:
        if len(seq) > n_res:
            seq = seq[:n_res]
        else:
            seq = seq + 'X' * (n_res - len(seq))

    aatype = sequence_to_aatype(seq)

    # Single rep from pair (mean pool + pad to 384)
    single = pair.mean(axis=1)
    if single.shape[1] < 384:
        single = np.concatenate([single, np.zeros((n_res, 384 - single.shape[1]), dtype=single.dtype)], axis=1)
    else:
        single = single[:, :384]

    # To tensors with batch dim
    evoformer_output = {
        "single": torch.from_numpy(single[None]).float().to(DEVICE),
        "pair": torch.from_numpy(pair[None]).float().to(DEVICE),
    }
    aatype_t = torch.from_numpy(aatype[None]).long().to(DEVICE)
    mask_t = torch.ones((1, n_res), dtype=torch.float32, device=DEVICE)

    with torch.no_grad():
        sm_out = model.structure_module(
            evoformer_output, aatype_t, mask=mask_t, inplace_safe=False,
        )

    # Convert to atom37 for PDB
    atom14_pos = sm_out["positions"][-1]
    feats = {
        "residx_atom37_to_atom14": torch.from_numpy(
            np.take(residue_constants.RESTYPE_ATOM37_TO_ATOM14, aatype, axis=0)
        ).long().to(DEVICE).unsqueeze(0),
        "atom37_atom_exists": torch.from_numpy(
            np.take(residue_constants.RESTYPE_ATOM37_MASK, aatype, axis=0)
        ).float().to(DEVICE).unsqueeze(0),
    }
    atom37_pos = atom14_to_atom37(atom14_pos, feats)
    atom37_mask = feats["atom37_atom_exists"]

    pdb_str = protein.to_pdb(
        protein.from_prediction(
            {"aatype": aatype[None], "residue_index": np.arange(n_res)[None]},
            {"final_atom_positions": atom37_pos[0].cpu().numpy(),
             "final_atom_mask": atom37_mask[0].cpu().numpy()},
        )
    )

    os.makedirs(output_dir, exist_ok=True)
    basename = os.path.splitext(os.path.basename(npy_path))[0]
    out_path = os.path.join(output_dir, f"{basename}_structure.pdb")
    with open(out_path, 'w') as f:
        f.write(pdb_str)
    return out_path


def parse_protein_name(filename):
    m = re.match(r'([0-9a-zA-Z]{4})_([A-Za-z])_pair_block_47\.npy', filename)
    if m:
        return m.group(1), m.group(2)
    m = re.match(r'([0-9a-zA-Z]{4})_([A-Za-z])', filename)
    if m:
        return m.group(1), m.group(2)
    return None, None


def process_directory(input_dir, output_dir):
    """Process all .npy files in a directory."""
    input_dir = str(input_dir)
    output_dir = str(output_dir)
    npy_files = sorted(f for f in os.listdir(input_dir) if f.endswith('.npy'))
    print(f"Processing {len(npy_files)} files: {input_dir} -> {output_dir}")
    
    succeeded, failed = 0, 0
    for npy_name in npy_files:
        pdb_id, chain_id = parse_protein_name(npy_name)
        if not pdb_id:
            print(f"  SKIP {npy_name}")
            failed += 1
            continue
        try:
            out = npy_to_pdb(os.path.join(input_dir, npy_name), output_dir, pdb_id, chain_id)
            print(f"  OK {npy_name} -> {os.path.basename(out)}")
            succeeded += 1
        except Exception as e:
            print(f"  FAIL {npy_name}: {e}")
            failed += 1
    
    print(f"\nDone: {succeeded} succeeded, {failed} failed")
    return succeeded, failed

print("Helper functions defined.")

## 4. Generate PDBs from ORIGINAL test proteins

In [ ]:
process_directory(ORIGINAL_NPY_DIR, ORIGINAL_PDB_DIR)

## 5. Generate PDBs from RECONSTRUCTED test proteins

In [ ]:
process_directory(RECON_NPY_DIR, RECON_PDB_DIR)

## 6. Verify PDB outputs

In [ ]:
orig_pdbs = sorted(ORIGINAL_PDB_DIR.glob('*.pdb')) if ORIGINAL_PDB_DIR.exists() else []
recon_pdbs = sorted(RECON_PDB_DIR.glob('*.pdb')) if RECON_PDB_DIR.exists() else []

print(f"Original PDBs:      {len(orig_pdbs)} files")
print(f"Reconstructed PDBs: {len(recon_pdbs)} files")

if orig_pdbs:
    print(f"\nFirst 5 original: {[p.name for p in orig_pdbs[:5]]}")
if recon_pdbs:
    print(f"First 5 reconstructed: {[p.name for p in recon_pdbs[:5]]}")

## 7. Compute TM-scores (reconstructed vs original)

This compares the PDB structures generated from reconstructed pair reps against
the PDB structures from original pair reps. High TM-scores (close to 1.0) mean
the SAE reconstruction preserved structural information well.

In [ ]:
import subprocess
from datetime import datetime

# Find TMalign
TMALIGN = None
for candidate in [
    str(VIZFOLD / 'bin' / 'TMalign'),
    'TMalign',
]:
    try:
        subprocess.run([candidate], capture_output=True, timeout=5)
        TMALIGN = candidate
        break
    except (FileNotFoundError, subprocess.TimeoutExpired):
        if os.path.isfile(candidate):
            TMALIGN = candidate
            break

if not TMALIGN:
    print("WARNING: TMalign not found. Skipping TM-score computation.")
    print("  Build TMalign and place at bin/TMalign, or install it on PATH.")
else:
    print(f"TMalign: {TMALIGN}")
    
    tm_scores = {}
    
    for recon_pdb in sorted(RECON_PDB_DIR.glob('*_structure.pdb')):
        # Find matching original
        orig_pdb = ORIGINAL_PDB_DIR / recon_pdb.name
        if not orig_pdb.exists():
            print(f"  SKIP {recon_pdb.name}: no matching original")
            continue
        
        try:
            result = subprocess.run(
                [TMALIGN, str(recon_pdb), str(orig_pdb)],
                capture_output=True, text=True, timeout=120,
            )
            for line in result.stdout.splitlines():
                if 'TM-score=' in line:
                    parts = line.split()
                    try:
                        score = float(parts[1])
                        name = recon_pdb.stem.replace('_structure', '')
                        tm_scores[name] = score
                        print(f"  {name}: TM-score = {score:.4f}")
                        break
                    except (IndexError, ValueError):
                        continue
        except Exception as e:
            print(f"  FAIL {recon_pdb.name}: {e}")
    
    if tm_scores:
        avg = sum(tm_scores.values()) / len(tm_scores)
        median = sorted(tm_scores.values())[len(tm_scores) // 2]
        
        print(f"\n{'='*50}")
        print(f"  Proteins scored: {len(tm_scores)}")
        print(f"  Average TM-score:  {avg:.4f}")
        print(f"  Median TM-score:   {median:.4f}")
        print(f"  Min:               {min(tm_scores.values()):.4f}")
        print(f"  Max:               {max(tm_scores.values()):.4f}")
        print(f"{'='*50}")
        
        # Save results
        out_json = str(RECON_PDB_DIR / 'tm_scores.json')
        with open(out_json, 'w') as f:
            json.dump({
                'tm_scores': tm_scores,
                'num_proteins': len(tm_scores),
                'avg_tm_score': avg,
                'median_tm_score': median,
                'timestamp': datetime.utcnow().isoformat(),
            }, f, indent=2)
        print(f"\nSaved: {out_json}")
        
        out_csv = str(RECON_PDB_DIR / 'tm_scores.csv')
        with open(out_csv, 'w') as f:
            f.write('protein_id,tm_score\n')
            for pid, sc in sorted(tm_scores.items()):
                f.write(f'{pid},{sc:.6f}\n')
        print(f"Saved: {out_csv}")
    else:
        print("No TM-scores computed.")

## 8. Results summary

In [ ]:
results_json = RECON_PDB_DIR / 'tm_scores.json'
if results_json.exists():
    with open(results_json) as f:
        results = json.load(f)
    
    print(f"Proteins scored: {results['num_proteins']}")
    print(f"Average TM-score: {results['avg_tm_score']:.4f}")
    print(f"Median TM-score:  {results['median_tm_score']:.4f}")
    print()
    print(f"{'Protein':<40} {'TM-score':>10}")
    print('-' * 52)
    for pid, score in sorted(results['tm_scores'].items(), key=lambda x: -x[1]):
        print(f"{pid:<40} {score:>10.4f}")
else:
    print('No TM-score results found. Check the steps above for errors.')